# Pré-anotação de luvas com Grounding DINO (Colab, GPU)

Os datasets novos (`novos_unificado`) têm muita luva visível **sem caixa**. Pela auditoria, isso afeta cerca de 55% das imagens do epids, 28% do ppecv e 19% do sgogg. Isso ensina ao modelo que luva é fundo.

Este notebook roda o Grounding DINO, um detector que acha objetos pelo nome e não depende do nosso best.pt, em todas as imagens. Ele gera um CSV com:
- **gloves**: luvas detectadas onde ainda não existe caixa de luva. Viram caixas pré-anotadas (tag `dino-gloves`) para revisar no Roboflow.
- **goggles**: óculos detectados sem caixa. O DINO erra muito nessa classe, então isso vira só a tag `suspeita-goggles` para revisar, não caixa.

O notebook **não altera** os rótulos. A mesclagem é feita localmente com `scripts/aplicar_dino.py`.

**Antes de rodar:**
1. Suba `novos_unificado.zip` para `MyDrive/TCC/yolo_dataset/novos_unificado.zip`.
2. Em `Runtime > Change runtime type`, selecione **GPU**.
3. Rode as células em ordem (Executar tudo). Se a sessão cair, é só rodar tudo de novo: o que já foi processado fica salvo no Drive e é pulado.

**Depois:** baixe `MyDrive/TCC/dino/dino_deteccoes.csv` para `datasets/novos_unificado/dino_deteccoes.csv` no seu PC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/TCC/yolo_dataset/novos_unificado.zip'
OUT_DIR = '/content/drive/MyDrive/TCC/dino'
OUT_CSV = f'{OUT_DIR}/dino_deteccoes.csv'
FEITOS = f'{OUT_DIR}/dino_feitos.txt'  # imagens ja processadas (inclusive as sem deteccao), para retomar

import os
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
assert os.path.exists(ZIP_PATH), f'Zip nao encontrado em {ZIP_PATH} - o upload pro Drive terminou? Nome/pasta certos?'
!pip -q install -U "transformers>=4.51"
!unzip -q -o "{ZIP_PATH}" -d /content
ROOT = '/content/novos_unificado'
!ls {ROOT}

In [ ]:
import torch
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

assert torch.cuda.is_available(), 'Sem GPU: Runtime > Change runtime type > GPU'

# mesmo modelo, prompts e limiares da auditoria local (scripts/auditar_rotulos.py),
# onde a precisao de gloves foi conferida a olho em ~80-85%
MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
proc = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to('cuda').eval()

CLASSES = ['helmet', 'vest', 'gloves', 'boots', 'goggles']
PROMPTS = {'hard hat': 'helmet', 'safety vest': 'vest', 'glove': 'gloves', 'boot': 'boots', 'safety goggles': 'goggles'}
TEXTO = '. '.join(PROMPTS) + '.'
MANTER = {'gloves', 'goggles'}  # o DINO erra demais em vest/helmet/boots

SCORE_MIN = 0.35
IOU_MATCH = 0.3   # deteccao que sobrepoe caixa existente da mesma classe = ja anotado
IOU_DUP = 0.5     # o DINO as vezes repete a mesma luva
AREA_MAX = 0.25   # caixa de luva cobrindo 1/4 da imagem e erro
BATCH = 2         # com 8 estourou os 15 GB da T4 (imagens grandes aumentam o lote todo)

In [ ]:
import csv, glob
from PIL import Image
from tqdm.auto import tqdm


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / uniao if uniao > 0 else 0


def ler_rotulo(img_path):
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    caixas = []
    if os.path.exists(lbl):
        for linha in open(lbl):
            p = linha.split()
            if len(p) == 5:
                c, xc, yc, w, h = int(p[0]), *map(float, p[1:])
                caixas.append((CLASSES[c], (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)))
    return caixas


def classe_do_texto(t):
    # o DINO as vezes junta dois prompts ('hard hat safety goggles'); descarta esses
    achadas = [c for prompt, c in PROMPTS.items() if prompt in t]
    return achadas[0] if len(achadas) == 1 else None


def inferir(ims):
    inp = proc(images=ims, text=[TEXTO] * len(ims), return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        saida = model(**inp)
    # target_sizes (1, 1) -> caixas ja saem normalizadas (0..1), como no YOLO
    return proc.post_process_grounded_object_detection(
        saida, inp.input_ids, threshold=SCORE_MIN, text_threshold=0.25,
        target_sizes=[(1, 1)] * len(ims))


imgs = sorted(glob.glob(f'{ROOT}/*/images/*'))
feitos = set(open(FEITOS).read().split()) if os.path.exists(FEITOS) else set()
fila = [p for p in imgs if os.path.basename(p) not in feitos]
assert imgs, f'Nenhuma imagem em {ROOT} - o unzip funcionou?'
print(f'{len(imgs)} imagens, {len(feitos)} ja feitas, {len(fila)} na fila')

novo_csv = not os.path.exists(OUT_CSV)
with open(OUT_CSV, 'a', newline='') as f_csv, open(FEITOS, 'a') as f_feitos:
    out = csv.writer(f_csv)
    if novo_csv:
        out.writerow(['arquivo', 'split', 'classe', 'score', 'xc', 'yc', 'w', 'h'])
    for i in tqdm(range(0, len(fila), BATCH)):
        lote = fila[i:i + BATCH]
        ims = [Image.open(p).convert('RGB') for p in lote]
        try:
            res = inferir(ims)
        except torch.cuda.OutOfMemoryError:
            # lote pesado demais: libera a memoria e vai uma imagem por vez
            torch.cuda.empty_cache()
            res = [inferir([im])[0] for im in ims]
        for p, r in zip(lote, res):
            gt = ler_rotulo(p)
            aceitas = []
            textos = r.get('text_labels', r.get('labels'))
            ordem = sorted(zip(textos, r['scores'].tolist(), r['boxes'].tolist()), key=lambda x: -x[1])
            for txt, score, box in ordem:
                cls = classe_do_texto(txt)
                if cls not in MANTER:
                    continue
                box = [min(max(v, 0.0), 1.0) for v in box]
                if (box[2] - box[0]) * (box[3] - box[1]) > AREA_MAX:
                    continue
                if any(c == cls and iou(box, b) >= IOU_MATCH for c, b in gt):
                    continue
                if any(c == cls and iou(box, b) >= IOU_DUP for c, b in aceitas):
                    continue
                aceitas.append((cls, box))
                out.writerow([os.path.basename(p), p.split('/')[-3], cls, round(score, 3),
                              round((box[0] + box[2]) / 2, 6), round((box[1] + box[3]) / 2, 6),
                              round(box[2] - box[0], 6), round(box[3] - box[1], 6)])
            f_feitos.write(os.path.basename(p) + '\n')
        f_csv.flush()
        f_feitos.flush()

In [ ]:
import pandas as pd

df = pd.read_csv(OUT_CSV)
df['dataset'] = df['arquivo'].str.split('_').str[0]
print('imagens com deteccao nova, por dataset e classe:')
print(df.groupby(['dataset', 'classe'])['arquivo'].nunique().unstack(fill_value=0))
print('\ncaixas novas por classe:', df['classe'].value_counts().to_dict())
print(f'\nPronto. Baixe {OUT_CSV} para datasets/novos_unificado/dino_deteccoes.csv no PC.')